In [ ]:
# # 九曜：天墟 LF01 — 旁白／蕭曜霖 國漫配音版（Qwen3-TTS，Kaggle）第 2 版 2026-09-30
#
# 咖哩：要《鬥破蒼穹》那種國漫旁白。第 1 版「設計聲音 → Base 模型照唸」在照唸時錯字多、冒粵語 →
# 第 2 版直接用 VoiceDesign 唸台詞（每句 4 個亂數種子），提示詞改成預告片式史詩旁白（沙啞、氣聲、抑揚頓挫大）。
# 另外每個檔輸出「混音版」（低頻加強、壓縮、殘響），模擬動畫成片的旁白聲音。
# 由 Claude 經 Kaggle API 推送執行；MP3 以 base64 印進執行紀錄供 Claude 取回。


In [ ]:
import subprocess
print(subprocess.run("pip install -q -U qwen-tts soundfile 2>&1 | tail -3; nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)


In [ ]:
import json, os
VOICES = json.loads(r'''{"VO": {"name": "旁白", "prompt": "熱血玄幻國漫（3D修仙動畫）片頭的史詩旁白。成熟男聲約四十歲，低沉、沙啞、帶很重的氣聲和磁性顆粒感，像電影預告片的配音。標準普通話。抑揚頓挫很大：開頭壓低、關鍵字加重拉長、句尾下沉；詞與詞之間有戲劇性停頓，充滿張力和宿命感，像在揭開一個傳說。"}, "C05": {"name": "蕭曜霖", "prompt": "熱血玄幻國漫（3D修仙動畫）裡的強者教官。四十多歲男聲，低沉、沙啞、胸腔共鳴厚，帶顆粒感。標準普通話，專業配音演員。聲音不大卻壓迫感十足，冷峻、不容置疑，一字一頓，每個字都像落在地上，說完不留餘地。"}}''')
LINES = [["VO", "LF01-A01_旁白", "九曜界……天玄院。"], ["VO", "LF01-A01x_旁白長句", "這裡，是九曜界。天墟崩裂之後，萬族並起。而一切的開始，是一座名為——天玄院的學院。"], ["C05", "LF01-A10a_蕭曜霖", "不用等明天。"], ["C05", "LF01-A10b_蕭曜霖", "半刻鐘後，新生演武場集合。"]]
SEEDS = [11, 22, 33, 44]
OUT = "/kaggle/working/voice_guoman"; os.makedirs(OUT, exist_ok=True)


In [ ]:
# 直接用 VoiceDesign 唸台詞（不經 Base 照唸）；T4 → float32＋sdpa
import torch, soundfile as sf
from qwen_tts import Qwen3TTSModel
design = Qwen3TTSModel.from_pretrained("Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign", device_map="cuda:0", dtype=torch.float32, attn_implementation="sdpa")
for spk, name, text in LINES:
    for s in SEEDS:
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        wavs, sr = design.generate_voice_design(text=text, language="Chinese", instruct=VOICES[spk]["prompt"])
        w = wavs[0][: int(sr * 25)]
        sf.write(f"{OUT}/{name}_s{s}.wav", w, sr)
        print(name, f"s{s}", f"{len(wavs[0])/sr:.2f}s", text)


In [ ]:
# （第 2 版不用 Base 照唸）


In [ ]:
# 乾聲＋混音版 MP3，並以 base64 印進執行紀錄（Claude 從雲端抓不到 Output 檔）
import glob, base64, shutil
MIX = "equalizer=f=110:t=q:w=1:g=5,equalizer=f=3500:t=q:w=1.5:g=2,acompressor=threshold=-20dB:ratio=4:attack=5:release=120:makeup=4,aecho=0.8:0.6:60|120:0.25|0.15,loudnorm=I=-16"
dst = "/kaggle/working/voice_guoman_tiny"; os.makedirs(dst, exist_ok=True)
for w in sorted(glob.glob(f"{OUT}/*.wav")):
    b = f"{dst}/{os.path.basename(w)[:-4]}"
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-ac", "1", "-b:a", "64k", b + ".mp3"], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", w, "-af", MIX, "-ac", "1", "-b:a", "64k", b + "_混音.mp3"], check=True)
shutil.make_archive("/kaggle/working/LF01_voice_guoman_tiny", "zip", dst)
for m in sorted(glob.glob(f"{dst}/*.mp3")):
    print(f"@@MP3 {os.path.basename(m)} {base64.b64encode(open(m, 'rb').read()).decode()}")
